In [2]:
# %pip install -U pandas numpy scikit-learn matplotlib  # if needed

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV, LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# RMSE that works on old/new sklearn
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def show_scores(name, y_true, y_pred):
    print(f"{name} | R2={r2_score(y_true, y_pred):.3f}  RMSE={rmse(y_true, y_pred):.3f}  MAE={mean_absolute_error(y_true, y_pred):.3f}")

def cv_rmse(model, X, y, cv_splits=5):
    cv = KFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
    # use neg_mean_squared_error (works on older sklearn), then sqrt per fold
    mse = -cross_val_score(model, X, y, cv=cv, scoring='neg_mean_squared_error')
    return np.sqrt(mse).mean(), np.sqrt(mse).std()


In [3]:
DATA_PATH = "hypertension_dataset.csv"
df = pd.read_csv(DATA_PATH)
df.shape, df.columns.tolist()[:10]  # just peeking

((174982, 23),
 ['Country',
  'Age',
  'BMI',
  'Cholesterol',
  'Systolic_BP',
  'Diastolic_BP',
  'Smoking_Status',
  'Alcohol_Intake',
  'Physical_Activity_Level',
  'Family_History'])

In [4]:
TARGET_COL = "Systolic_BP"   # continuous target for linear models
assert TARGET_COL in df.columns, "TARGET_COL missing"

# auto split by dtype
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != TARGET_COL]

# (optional) drop very high-cardinality categoricals (keeps things manageable)
HIGH_CARD_THRESHOLD = 50
too_many = [c for c in cat_cols if df[c].nunique(dropna=True) > HIGH_CARD_THRESHOLD]
if too_many:
    print("Dropping high-cardinality categoricals (to keep the one-hot sane):", too_many)
    cat_cols = [c for c in cat_cols if c not in too_many]

print("categorical:", cat_cols)
print("numeric (first few):", num_cols[:8])

# basic NA handling for Week 2 (simple)
used = [TARGET_COL] + cat_cols + num_cols
work = df[used].dropna(axis=0, how='any').copy()

X = work[cat_cols + num_cols]
y = work[TARGET_COL]

X.shape, y.shape


categorical: ['Country', 'Smoking_Status', 'Physical_Activity_Level', 'Family_History', 'Diabetes', 'Gender', 'Education_Level', 'Employment_Status', 'Hypertension']
numeric (first few): ['Age', 'BMI', 'Cholesterol', 'Diastolic_BP', 'Alcohol_Intake', 'Stress_Level', 'Salt_Intake', 'Sleep_Duration']


((174982, 22), (174982,))

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# OneHot encoder (compat with older sklearn where sparse_output isn't available)
try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

scaler = StandardScaler(with_mean=True, with_std=True)

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', ohe, cat_cols),
        ('num', scaler, num_cols),
    ],
    remainder='drop'
)

# handy: get transformed feature names after fit (for coef inspection)
def get_feature_names(prep):
    names = []
    if cat_cols:
        ohe = prep.named_transformers_['cat']
        names += list(ohe.get_feature_names_out(cat_cols))
    if num_cols:
        names += num_cols
    return names


In [6]:
alphas = np.logspace(-3, 3, 13)  # 0.001 ... 1000

ridge = Pipeline([
    ('prep', preprocessor),
    ('model', RidgeCV(alphas=alphas, cv=5))
])

ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)

# chosen alpha lives in the step
chosen_alpha = ridge.named_steps['model'].alpha_
print("Ridge alpha*:", chosen_alpha)

show_scores("Ridge (test)", y_test, y_pred_ridge)

mean_cv, std_cv = cv_rmse(ridge, X, y, cv_splits=5)
print(f"Ridge CV RMSE mean±std: {mean_cv:.3f} ± {std_cv:.3f}")


Ridge alpha*: 1000.0
Ridge (test) | R2=-0.000  RMSE=26.096  MAE=22.623
Ridge CV RMSE mean±std: 26.023 ± 0.055


In [7]:
lasso = Pipeline([
    ('prep', preprocessor),
    ('model', LassoCV(alphas=alphas, cv=5, max_iter=10000, random_state=RANDOM_STATE))
])

lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)

chosen_alpha = lasso.named_steps['model'].alpha_
print("Lasso alpha*:", chosen_alpha)

show_scores("Lasso (test)", y_test, y_pred_lasso)

mean_cv, std_cv = cv_rmse(lasso, X, y, cv_splits=5)
print(f"Lasso CV RMSE mean±std: {mean_cv:.3f} ± {std_cv:.3f}")


Lasso alpha*: 1000.0
Lasso (test) | R2=-0.000  RMSE=26.093  MAE=22.623
Lasso CV RMSE mean±std: 26.020 ± 0.054


In [8]:
# try a small grid of l1_ratio (0 = ridge-like, 1 = lasso-like)
l1_grid = [0.2, 0.5, 0.8]

enet = Pipeline([
    ('prep', preprocessor),
    ('model', ElasticNetCV(l1_ratio=l1_grid, alphas=alphas, cv=5, max_iter=10000, random_state=RANDOM_STATE))
])

enet.fit(X_train, y_train)
y_pred_enet = enet.predict(X_test)

m = enet.named_steps['model']
print("ElasticNet alpha*:", m.alpha_, "| l1_ratio*:", m.l1_ratio_)

show_scores("ElasticNet (test)", y_test, y_pred_enet)

mean_cv, std_cv = cv_rmse(enet, X, y, cv_splits=5)
print(f"ElasticNet CV RMSE mean±std: {mean_cv:.3f} ± {std_cv:.3f}")


ElasticNet alpha*: 1000.0 | l1_ratio*: 0.2
ElasticNet (test) | R2=-0.000  RMSE=26.093  MAE=22.623
ElasticNet CV RMSE mean±std: 26.020 ± 0.054


In [9]:
def top_coefs(pipeline, topk=15):
    # fit the preprocessor on TRAIN so feature names align with model.coef_
    prep = pipeline.named_steps['prep']
    model = pipeline.named_steps['model']
    # Ensure preprocessor is fitted
    if not hasattr(prep, 'transform'):
        _ = pipeline.predict(X_test)  # forces fit during predict
    names = get_feature_names(prep)
    try:
        coefs = model.coef_.ravel()
    except:
        coefs = np.asarray(model.coef_).ravel()
    dfc = pd.DataFrame({"feature": names, "coef": coefs, "abscoef": np.abs(coefs)})
    return dfc.sort_values("abscoef", ascending=False).head(topk)

print("Top Ridge features:")
display(top_coefs(ridge, 15))

print("Top Lasso features:")
display(top_coefs(lasso, 15))

print("Top ElasticNet features:")
display(top_coefs(enet, 15))


Top Ridge features:


,feature,coef,abscoef
14,Country_South Africa,0.378247,0.378247
11,Country_Mexico,-0.285884,0.285884
15,Country_South Korea,0.265433,0.265433
37,Employment_Status_Unemployed,-0.234292,0.234292
9,Country_Italy,0.221522,0.221522
35,Employment_Status_Employed,0.211817,0.211817
0,Country_Argentina,-0.205607,0.205607
18,Country_UK,-0.198485,0.198485
12,Country_Russia,-0.182807,0.182807
5,Country_France,-0.162281,0.162281


Top Lasso features:


,feature,coef,abscoef
0,Country_Argentina,-0.0,0.0
27,Family_History_Yes,-0.0,0.0
29,Diabetes_Yes,0.0,0.0
30,Gender_Female,-0.0,0.0
31,Gender_Male,0.0,0.0
32,Education_Level_Primary,0.0,0.0
33,Education_Level_Secondary,-0.0,0.0
34,Education_Level_Tertiary,0.0,0.0
35,Employment_Status_Employed,0.0,0.0
36,Employment_Status_Retired,0.0,0.0


Top ElasticNet features:


,feature,coef,abscoef
0,Country_Argentina,-0.0,0.0
27,Family_History_Yes,-0.0,0.0
29,Diabetes_Yes,0.0,0.0
30,Gender_Female,-0.0,0.0
31,Gender_Male,0.0,0.0
32,Education_Level_Primary,0.0,0.0
33,Education_Level_Secondary,-0.0,0.0
34,Education_Level_Tertiary,0.0,0.0
35,Employment_Status_Employed,0.0,0.0
36,Employment_Status_Retired,0.0,0.0


In [10]:
notes = {
    "what_i_did": [
        "Used Ridge/Lasso/ElasticNet with 5-fold CV inside sklearn's *CV models*.",
        "Preprocessed with one-hot for categoricals and scaling for numerics."
    ],
    "hyperparameters": [
        "Ridge alpha chosen by CV from logspace(1e-3..1e3).",
        "Lasso alpha chosen by CV; many small coefficients got shrunk to 0.",
        "ElasticNet chose both alpha and l1_ratio from a small grid."
    ],
    "metrics_takeaway": [
        "Compare R2/RMSE/MAE across models; keep the one that generalizes best (CV RMSE and test RMSE)."
    ],
    "collinearity_note": [
        "Regularization helped tame multicollinearity I saw last week; Ridge spreads weights, Lasso zeros some.",
    ],
    "next_steps": [
        "Try a bigger l1_ratio grid; check stability across random seeds.",
        "Consider dropping very high-cardinality features or target-transform (if residuals are skewed)."
    ]
}
for k, v in notes.items():
    print(k.upper()+":")
    for item in v:
        print(" -", item)
    print()


WHAT_I_DID:
 - Used Ridge/Lasso/ElasticNet with 5-fold CV inside sklearn's *CV models*.
 - Preprocessed with one-hot for categoricals and scaling for numerics.

HYPERPARAMETERS:
 - Ridge alpha chosen by CV from logspace(1e-3..1e3).
 - Lasso alpha chosen by CV; many small coefficients got shrunk to 0.
 - ElasticNet chose both alpha and l1_ratio from a small grid.

METRICS_TAKEAWAY:
 - Compare R2/RMSE/MAE across models; keep the one that generalizes best (CV RMSE and test RMSE).

COLLINEARITY_NOTE:
 - Regularization helped tame multicollinearity I saw last week; Ridge spreads weights, Lasso zeros some.

NEXT_STEPS:
 - Try a bigger l1_ratio grid; check stability across random seeds.
 - Consider dropping very high-cardinality features or target-transform (if residuals are skewed).

